In [ ]:
! pip install numpyro healpy einops reproject

In [2]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
import os
os.environ["XLA_FLAGS"] = "--xla_gpu_force_compilation_parallelism=1"

# from google.colab import drive
# drive.mount('/content/drive')
# sys.path.append(r'/content/drive/Othercomputers/My MacBook Pro/fermi-prob-prog/')
# %cd /content/drive/Othercomputers/My MacBook Pro/fermi-prob-prog/notebooks

import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist

import numpy as np
import arviz as az
import healpy as hp
import pickle
from tqdm import tqdm

print(jax.devices())

/n/home07/yitians/.conda/envs/torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[gpu(id=0)]


In [3]:
save_dir = "../outputs/poisson_sim/run_230718"
from models.np_model import NPModel
npmodel = NPModel(
    non_poissonian=True,
    l_max=2,
    vary_gamma=True,
    bulge_hybrid=True,
    ps_cat="3fgl",
    nside=128,
)

Loading the psf correction from: /n/home07/yitians/fermi/fermi-prob-prog/notebooks/psf_dir/Fermi_PSF_2GeV2_nside128.npy
Max photon count is 103


In [ ]:
for i in range(74, 80):
    counts = jnp.array(np.load(f"{save_dir}/counts_{i}.npy"), dtype=jnp.int32)
    svi_results = npmodel.fit_svi(
        rng_key=jax.random.PRNGKey(4234),
        n_steps=2000,
        guide="iaf",
        lr=5e-5,
        num_particles=8,
        data=jnp.array(counts),
    )
    pickle.dump(svi_results, open(f"{save_dir}/svi_results_{i}.p", 'wb'))
    samples = npmodel.get_svi_samples(
        rng_key=jax.random.PRNGKey(42),
        num_samples=50000,
    )
    pickle.dump(samples, open(f"{save_dir}/samples_{i}.p", 'wb'))